# 05 – Clustering: Segmentierung der Antragsteller

**Unüberwachtes** Verfahren (Teilfrage 4). **TARGET wird NICHT verwendet** – sonst wäre
die Segmentierung zirkulär. Die Ausfallrate je Cluster betrachten wir erst NACHTRÄGLICH.

**Warum Clustering überhaupt:** Es deckt Struktur/Gruppen auf, die für Risikomanagement,
Produktgestaltung oder gezielte (faire!) Beratung nützlich sein können – unabhängig von
der reinen Ausfallvorhersage.

In [4]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, utils, clustering as cl
feat = utils.load_processed("feature_matrix")

## 1. Feature-Auswahl (ohne TARGET) & Skalierung

**Warum diese Merkmale:** decken Einkommen, Verschuldung, Alter, Bonität, Historie und
Zahlungsverhalten ab. **Warum Skalierung zwingend:** K-Means/PCA sind distanz-/varianz-
basiert; ohne Standardisierung dominieren großskalige Merkmale.

In [5]:
X = cl.select_cluster_features(feat)
pre = cl.build_clustering_preprocessor()
X_scaled = pre.fit_transform(X)
print("Clustering-Matrix:", X_scaled.shape)

18:39:28 | INFO    | Clustering nutzt 12 Merkmale: ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'credit_income_ratio', 'annuity_income_ratio', 'age_years', 'employment_years', 'external_score_mean', 'BUREAU_COUNT', 'BUREAU_ACTIVE_COUNT', 'PREV_COUNT', 'INST_LATE_PAYMENT_RATIO', 'missing_values_count']


Clustering-Matrix: (307511, 12)


## 2. Wahl der Clusterzahl: Elbow & Silhouette

**Interpretation:** Elbow sucht den "Knick" der Inertia; Silhouette bewertet die
Trennschärfe (höher = besser). **Grenze:** Heuristiken; niedrige Silhouette-Werte sind
bei realen, verrauschten Daten üblich und bedeuten, dass die Gruppen *graduell*
übergehen, nicht scharf getrennt sind.

In [6]:
best_k, inertias, silhouettes = cl.find_optimal_k(X_scaled, k_range=range(2, 9))
plt.show()
print("Vorgeschlagenes k (Silhouette):", best_k)

19:02:50 | INFO    | k=2 | Inertia=3234816 | Silhouette=0.1460
19:23:31 | INFO    | k=3 | Inertia=2923329 | Silhouette=0.1406
19:43:00 | INFO    | k=4 | Inertia=2680696 | Silhouette=0.1407
20:00:51 | INFO    | k=5 | Inertia=2493126 | Silhouette=0.1097
20:18:02 | INFO    | k=6 | Inertia=2351221 | Silhouette=0.1128
21:36:16 | INFO    | k=7 | Inertia=2252293 | Silhouette=0.1031
21:53:36 | INFO    | k=8 | Inertia=2132296 | Silhouette=0.1081
21:53:37 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\clustering\cluster_k_selection.png
21:53:37 | INFO    | Bester k nach Silhouette: 2


Vorgeschlagenes k (Silhouette): 2


C:\Users\annis\AppData\Local\Temp\ipykernel_29568\2732511711.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. K-Means & PCA-Visualisierung

**Warum PCA:** projiziert die hochdimensionalen Cluster auf 2D zur visuellen Prüfung.
**Grenze:** die 2 Komponenten erklären nur einen Teil der Varianz (im Achsentitel
angegeben) – Überlappung in 2D ≠ Überlappung im vollen Raum.

In [7]:
# Wir fixieren k bewusst auf einen fachlich interpretierbaren Wert (hier 4),
# da Silhouette und fachliche Interpretierbarkeit gegeneinander abzuwägen sind.
K = 4
labels, profiles, km, pre = cl.run_clustering(feat, k=K)
plt.show()

21:53:37 | INFO    | Clustering nutzt 12 Merkmale: ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'credit_income_ratio', 'annuity_income_ratio', 'age_years', 'employment_years', 'external_score_mean', 'BUREAU_COUNT', 'BUREAU_ACTIVE_COUNT', 'PREV_COUNT', 'INST_LATE_PAYMENT_RATIO', 'missing_values_count']
21:53:45 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\clustering\clusters_pca_2d.png
21:53:46 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\clustering\cluster_profiles.csv
21:53:46 | INFO    | Clustering fertig: k=4, erklärte Varianz (2 PCs)=35.2%
C:\Users\annis\AppData\Local\Temp\ipykernel_29568\3644668956.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Cluster-Profile & Ausfallrate je Cluster

**Was:** durchschnittliche Merkmalswerte, Clustergröße und – nachträglich – die
Ausfallrate je Cluster. **Interpretation:** Unterscheiden sich die Ausfallraten,
ist die merkmalsbasierte Segmentierung risikorelevant. **Grenze:** rein deskriptiv,
**nicht kausal** – ein Cluster "verursacht" keinen Ausfall.

In [8]:
show_cols = ["cluster", "cluster_size", "default_rate", "AMT_INCOME_TOTAL",
             "credit_income_ratio", "external_score_mean", "age_years",
             "INST_LATE_PAYMENT_RATIO"]
show_cols = [c for c in show_cols if c in profiles.columns]
profiles[show_cols].round(3)

,cluster,cluster_size,default_rate,AMT_INCOME_TOTAL,credit_income_ratio,external_score_mean,age_years,INST_LATE_PAYMENT_RATIO
0,0,80761,0.062,1.531839e+05,7.247,0.552,46.668,0.066
1,1,69808,0.089,2.016560e+05,3.113,0.483,45.308,0.081
2,2,156941,0.086,1.614730e+05,2.641,0.499,41.863,0.078
3,3,1,1.000,1.170000e+08,0.005,0.240,34.500,0.375


## 5. Fachliche Cluster-Interpretation (Beispiel-Lesart)

Je nach Profil lassen sich Cluster benennen, z. B.: *einkommensstark/niedrig verschuldet*,
*hohe Kredit-Einkommens-Relation*, *viele Voranträge*, *auffällige Zahlungshistorie*,
*viele fehlende Informationen*. Diese Etiketten sind **interpretativ** und müssen am
konkreten (echten) Profil belegt werden.

**Grenzen des Clusterings:** Ergebnis hängt von Feature-Auswahl, Skalierung und k ab;
K-Means unterstellt etwa kugelförmige, ähnlich große Cluster. Alternativen (GMM,
hierarchisch) können andere Strukturen finden.

---
### Quellen (im Theorieteil vollständig)
- Jain, A. K. (2010). Data clustering: 50 years beyond K-means. *Pattern Recognition Letters.*
- Rousseeuw, P. J. (1987). Silhouettes. *J. Computational and Applied Mathematics.*